# Objective
Assess whether the company's rapid client growth (2021–2025) is profitable and sustainable, by analysing revenue/cost trends, client segments, and satisfaction drivers to recommend where to focus retention and pricing efforts.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
import numpy as np

In [2]:
profiles = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client profiles.xlsx")
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR
0,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,More than 200,Finance
1,G1-002,Govt,Yes,NaN,,2020-05-19 00:00:00,More than 200,Security
2,G1-003,Govt,Yes,NaN,,2021-07-18 00:00:00,> 200,Transportation
3,G1-005,Govt,Y,NaN,NaN,2022-02-11 00:00:00,> 200,Security
4,G1-006,Govt,Y,NaN,NaN,2022-03-18 00:00:00,50 ~ 200,Legal


Things to clean:\
    - columns using different values to represent the same info(Sg-based,staff strength)\
    - getting rid of missing values (Country of overseas client)

In [3]:
profiles[['COUNTRY OF OVERSEAS CLIENT','OVERSEAS','SG-BASED']].value_counts()

COUNTRY OF OVERSEAS CLIENT  OVERSEAS  SG-BASED
Malaysia                    Yes                   6
                            Y                     3
India                       Yes                   1
China                       Y                     1
Indonesia                   Yes                   1
Malaysian                   Y                     1
Name: count, dtype: int64

In [4]:
# ---- Clean Client Profiles ----
# Strip stray leading/trailing whitespace on text columns
obj_cols = ['SG-BASED', 'OVERSEAS', 'COUNTRY OF OVERSEAS CLIENT',
            'STAFF STRENGTH', 'SECTOR', 'TYPE', 'CLIENT ID']
for c in obj_cols:
    profiles[c] = profiles[c].apply(lambda v: v.strip() if isinstance(v, str) else v)

# SG-BASED / OVERSEAS: standardise to explicit Y / N ('Yes' and 'Y' -> 'Y', blanks -> 'N')
def yes_no(v):
    return 'Y' if str(v).strip().upper() in ('Y', 'YES') else 'N'
profiles['SG-BASED'] = profiles['SG-BASED'].apply(yes_no)
profiles['OVERSEAS'] = profiles['OVERSEAS'].apply(yes_no)

# COUNTRY OF OVERSEAS CLIENT: unify blanks to NaN, fix 'Malaysian' -> 'Malaysia'
profiles['COUNTRY OF OVERSEAS CLIENT'] = (
    profiles['COUNTRY OF OVERSEAS CLIENT']
    .replace(r'^\s*$', np.nan, regex=True)       # empty / whitespace-only -> NaN
    .replace('Malaysian', 'Malaysia')
)

# Derived country: domestic clients -> 'Singapore', otherwise the overseas country
profiles['country'] = profiles['COUNTRY OF OVERSEAS CLIENT'].fillna('Singapore')

# STAFF STRENGTH: normalise synonym labels, then derive an org_size bucket.
# Metadata rule: 1~49 = Small, 50~200 = Medium, >200 = Large.
# 'Less than 200' is ambiguous (spans Small & Medium) -> kept as 'Unspecified'.
profiles['STAFF STRENGTH'] = profiles['STAFF STRENGTH'].replace(
    {'More than 200': '> 200', 'Less than 49': '1 ~ 49'})
size_map = {'1 ~ 49': 'Small', '50 ~ 200': 'Medium',
            '> 200': 'Large', 'Less than 200': 'Unspecified'}
profiles['org_size'] = profiles['STAFF STRENGTH'].map(size_map)
assert profiles['org_size'].notna().all(), \
    profiles.loc[profiles['org_size'].isna(), 'STAFF STRENGTH'].unique()

# COMMENCEMENT DATE: mix of datetime & string -> uniform datetime64
profiles['COMMENCEMENT DATE'] = pd.to_datetime(profiles['COMMENCEMENT DATE'], errors='coerce')
assert profiles['COMMENCEMENT DATE'].notna().all(), "Unparseable COMMENCEMENT DATE remains"

profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,country,org_size
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [5]:
profiles['STAFF STRENGTH'].value_counts()

STAFF STRENGTH
> 200            62
50 ~ 200         57
1 ~ 49           22
Less than 200     7
Name: count, dtype: int64

In [6]:
profiles['SG-BASED'].value_counts()

SG-BASED
Y    114
N     34
Name: count, dtype: int64

In [7]:
profiles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   CLIENT ID                   148 non-null    object        
 1   TYPE                        148 non-null    object        
 2   SG-BASED                    148 non-null    object        
 3   OVERSEAS                    148 non-null    object        
 4   COUNTRY OF OVERSEAS CLIENT  34 non-null     object        
 5   COMMENCEMENT DATE           148 non-null    datetime64[ns]
 6   STAFF STRENGTH              148 non-null    object        
 7   SECTOR                      148 non-null    object        
 8   country                     148 non-null    object        
 9   org_size                    148 non-null    object        
dtypes: datetime64[ns](1), object(9)
memory usage: 11.7+ KB


In [8]:
survey = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client survey.xlsx")
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING
0,2021,G1-001,3,4,4,5,8
1,2021,G1-002,3,4,4,2,7
2,2021,G1-003,5,4,3,5,8
3,2021,G1-001,3,4,4,5,8
4,2021,G1-002,3,4,4,2,7


In [9]:
survey['CLIENT ID'].value_counts()

CLIENT ID
G1-001    9
G1-002    9
G1-003    9
G1-006    7
N1-001    5
         ..
P3-027    1
P3-028    1
P3-029    1
P3-030    1
P3-031    1
Name: count, Length: 143, dtype: int64

In [10]:
survey.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432 entries, 0 to 431
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   YEAR                      432 non-null    int64 
 1   CLIENT ID                 432 non-null    object
 2   PRESALES AND PARTNERSHIP  432 non-null    int64 
 3   TECHNICAL EXPERTISE       432 non-null    int64 
 4   PROJECT DELIVERY          432 non-null    int64 
 5   POST-SALES SUPPORT        432 non-null    int64 
 6   NPS RATING                432 non-null    int64 
dtypes: int64(6), object(1)
memory usage: 23.8+ KB


In [11]:
survey.isnull().sum()

YEAR                        0
CLIENT ID                   0
PRESALES AND PARTNERSHIP    0
TECHNICAL EXPERTISE         0
PROJECT DELIVERY            0
POST-SALES SUPPORT          0
NPS RATING                  0
dtype: int64

In [12]:
# ---- Clean Client Survey ----
# Drop 30 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
survey = survey.drop_duplicates()
survey = survey[survey['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived NPS category per respondent (Meta Data thresholds)
def nps_category(score):
    if score >= 9:
        return 'Promoter'    # 9-10
    elif score >= 7:
        return 'Passive'     # 7-8
    return 'Detractor'       # 6 or less
survey['nps_category'] = survey['NPS RATING'].apply(nps_category)

survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,nps_category
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [13]:
RC = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Revenue and COGS.xlsx")
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,2021,148180,17321,25562,41201
1,G1-001,2022,197313,23604,32875,50085
2,G1-001,2023,349899,51332,71695,89089
3,G1-001,2024,268821,40969,56452,76545
4,G1-001,2025,138115,14516,20889,28968


In [14]:
# ---- Clean Revenue and COGS ----
# Drop 21 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
# (P1-010, P1-018, P1-035 -> metadata says such records can be ignored)
RC = RC.drop_duplicates()
RC = RC[RC['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived cost / profitability metrics (formulas from Meta Data)
RC['cogs'] = RC['HARDWARE'] + RC['SOFTWARE'] + RC['MANPOWER']
RC['gross_profit'] = RC['REVENUE'] - RC['cogs']
RC['gross_margin_pct'] = (RC['gross_profit'] / RC['REVENUE'] * 100).round(2)
RC['gross_markup_pct'] = (RC['gross_profit'] / RC['cogs'] * 100).round(2)
RC['loss'] = RC['gross_profit'] < 0   # negative gross profit (all NPO clients) -> flagged, kept

RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,cogs,gross_profit,gross_margin_pct,gross_markup_pct,loss
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [15]:
RC.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CLIENT ID         392 non-null    object 
 1   YEAR              392 non-null    int64  
 2   REVENUE           392 non-null    int64  
 3   HARDWARE          392 non-null    int64  
 4   SOFTWARE          392 non-null    int64  
 5   MANPOWER          392 non-null    int64  
 6   cogs              392 non-null    int64  
 7   gross_profit      392 non-null    int64  
 8   gross_margin_pct  392 non-null    float64
 9   gross_markup_pct  392 non-null    float64
 10  loss              392 non-null    bool   
dtypes: bool(1), float64(2), int64(7), object(1)
memory usage: 31.1+ KB


In [16]:
RC.isnull().sum()

CLIENT ID           0
YEAR                0
REVENUE             0
HARDWARE            0
SOFTWARE            0
MANPOWER            0
cogs                0
gross_profit        0
gross_margin_pct    0
gross_markup_pct    0
loss                0
dtype: int64

In [17]:
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,country,org_size
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [18]:
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,nps_category
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [19]:
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,cogs,gross_profit,gross_margin_pct,gross_markup_pct,loss
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [20]:
# ---- Export cleaned datasets ----
import os
os.makedirs('cleaned', exist_ok=True)
profiles.to_csv('cleaned/client_profiles_clean.csv', index=False)
RC.to_csv('cleaned/revenue_cogs_clean.csv', index=False)
survey.to_csv('cleaned/client_survey_clean.csv', index=False)
print('Exported cleaned files:', os.listdir('cleaned'))

Exported cleaned files: ['client_profiles_clean.csv', 'client_survey_clean.csv', 'revenue_cogs_clean.csv']


## DB 1 (Elliot)

#### Graph 1

In [21]:
client_year_df = (
    RC
    .merge(profiles, on="CLIENT ID", how="left")
    .merge(survey, on=["CLIENT ID", "YEAR"], how="inner")
)

In [22]:
nps_categories = ["Detractor", "Passive", "Promoter"]              # legend order
category_colours = {"Detractor": "#d62728", "Passive": "#9aa0a6", "Promoter": "#2ca02c"}

# One least-squares line per YEAR, in plain pandas:
#   slope = cov(x, y) / var(x),  intercept = mean(y) - slope * mean(x)
# Two endpoints are enough to draw a straight line.
def yearly_trend(year_group):
    slope = year_group["NPS RATING"].cov(year_group["gross_margin_pct"]) / year_group["NPS RATING"].var()
    intercept = year_group["gross_margin_pct"].mean() - slope * year_group["NPS RATING"].mean()
    x = pd.Series([4, 10])                                        # span the full NPS scale
    return pd.DataFrame({"NPS RATING": x, "gross_margin_pct": intercept + slope * x})

trend = (client_year_df.groupby("YEAR")[["NPS RATING", "gross_margin_pct"]]
           .apply(yearly_trend)
           .reset_index())
trend["SERIES"] = "Trend for this year"
trend

,YEAR,level_1,NPS RATING,gross_margin_pct,SERIES
0,2021,0,4,43.593982,Trend for this year
1,2021,1,10,42.722741,Trend for this year
2,2022,0,4,48.722021,Trend for this year
3,2022,1,10,42.916064,Trend for this year
4,2023,0,4,44.710546,Trend for this year
5,2023,1,10,38.738449,Trend for this year
6,2024,0,4,46.965845,Trend for this year
7,2024,1,10,39.626624,Trend for this year
8,2025,0,4,50.496931,Trend for this year
9,2025,1,10,39.446947,Trend for this year


In [23]:
fig = px.scatter(
    client_year_df,
    x="NPS RATING",
    y="gross_margin_pct",
    size="REVENUE",                       # bubble size = deal size
    color="nps_category",
    color_discrete_map=category_colours,  # red = detractor, grey = passive, green = promoter
    category_orders={"nps_category": nps_categories},
    animation_frame="YEAR",               # -> year slider + play button
    animation_group="CLIENT ID",          # same client morphs smoothly between years
    hover_name="CLIENT ID",
    hover_data={"SECTOR": True, "TYPE": True, "org_size": True,
                "REVENUE": ":,.0f", "gross_margin_pct": ":.1f"},
    size_max=45,
    range_x=[3.5, 10.5],                 # fixed axes so the animation doesn't jump
    range_y=[-40, 70],
    title="Do happier client earn bring in more profit?",
)

# Animated trendline: px.line gives us one line trace per year, which we splice into
# the scatter's frames so the fit is redrawn every time the year slider moves.
trend_fig = px.line(trend, x="NPS RATING", y="gross_margin_pct",
                    animation_frame="YEAR", color="SERIES",
                    color_discrete_sequence=["#333"])

fig.add_traces(trend_fig.data)                     # line for the year the slider starts on
lines_by_year = {frame.name: frame.data for frame in trend_fig.frames}
for frame in fig.frames:                           # ...and for every other year
    frame.data = frame.data + lines_by_year[frame.name]

# Reference lines
overall_margin = client_year_df["gross_profit"].sum() / client_year_df["REVENUE"].sum() * 100
fig.add_hline(y=overall_margin, line_dash="dot", line_color="#555",
              annotation_text=f"Overall margin {overall_margin:.1f}%",
              annotation_position="top left")
fig.add_hline(y=0, line_color="#d62728", line_width=1,
              annotation_text="Break-even", annotation_position="bottom right")

fig.update_layout(
    xaxis_title="NPS Rating (0-10)",
    yaxis_title="Gross Margin (%)",
    template="plotly_white",
    height=700,
)

fig.show()

### Insights

1. Happier clients = thinner margins. Gross margin falls from 46.6% (Detractors) -> 43.4% (Passives) -> 40.6% (Promoters), even though Promoters bring the highest revenue (~$390k vs ~$279k).
2. The gap is widening every year. Trend line was flat in 2021 (−0.15% margin per NPS point) but steepens to −1.85% by 2025. Top-rated clients now earn ~10% less margin than low-rated ones.
3. All 5 loss-making engagements are high-NPS charity clients. They rate us 9+ while we lose money having them as cilents (e.g. N1-002, N1-007 for two years running).

#### Graph 2

In [24]:
cogs_df = RC.merge(profiles, on="CLIENT ID", how="left")

cogs_components = ["HARDWARE", "SOFTWARE", "MANPOWER"]                       # dataset field names
component_colours = {"HARDWARE": "#8c6d31", "SOFTWARE": "#6b6ecf", "MANPOWER": "#e6550d"}
client_type_options = ["All clients", "Private", "Govt", "NPO"]

In [25]:
def yearly_costs(data):
    """Yearly totals per COGS component, weighted margin, and cost shares."""
    yearly = data.groupby("YEAR")[cogs_components + ["REVENUE", "gross_profit"]].sum().reset_index()
    yearly["margin"] = yearly["gross_profit"] / yearly["REVENUE"] * 100
    cogs_total = yearly[cogs_components].sum(axis=1)
    for component in cogs_components:
        yearly[component + " share"] = yearly[component] / cogs_total * 100   # used in hover text
    return yearly

yearly_by_type = {
    client_type: yearly_costs(cogs_df if client_type == "All clients" else cogs_df[cogs_df["TYPE"] == client_type])
    for client_type in client_type_options
}

In [26]:
fig = go.Figure()

traces_per_type = len(cogs_components) + 1                  # 3 cost bars + 1 margin line

for client_type in client_type_options:
    yearly = yearly_by_type[client_type]
    show = client_type == client_type_options[0]
    for component in cogs_components:
        fig.add_trace(go.Bar(
            x=yearly["YEAR"], y=yearly[component],
            name=component.title(),
            marker_color=component_colours[component],
            visible=show,
            legendgroup=component,                       # one legend entry per component
            customdata=yearly[component + " share"],
            hovertemplate=component.title() + ": $%{y:,.0f} (%{customdata:.0f}% of COGS)",
        ))
    fig.add_trace(go.Scatter(
        x=yearly["YEAR"], y=yearly["margin"],
        name="Gross Margin %",
        mode="lines+markers+text",
        text=yearly["margin"].round(1).astype(str) + "%",
        textposition="top center",
        line=dict(color="#222", width=3),
        marker=dict(size=9),
        visible=show,
        yaxis="y2",                              # secondary axis: % not $
        hovertemplate="Margin: %{y:.1f}%",
    ))

# Dropdown to switch which client type's traces are visible (title updates to match).
buttons = [dict(label=selected_type, method="update",
                args=[{"visible": [trace_type == selected_type
                                    for trace_type in client_type_options
                                    for _ in range(traces_per_type)]},
                      {"title.text": f"COGS composition and gross margin for {selected_type}"}])
           for selected_type in client_type_options]

fig.update_layout(
    barmode="stack",
    title=f"COGS composition and gross margin for {client_type_options[0]}",
    xaxis=dict(title="Year", tickmode="linear", dtick=1),
    yaxis=dict(title="Cost of Goods Sold ($)", tickformat="~s"),   # 10M, 20M ...
    yaxis2=dict(title="Gross Margin (%)", overlaying="y", side="right",
                range=[0, 60], showgrid=False),
    updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right",
                      y=1.15, yanchor="top")],
    hovermode="x",
    legend=dict(orientation="h", y=-0.18),
    template="plotly_white",
    height=600,
    margin=dict(t=90),
)

fig.show()

### Insights
1. Manpower is the biggest cost every year, and it keeps growing. From 41% of COGS in 2021 to 44% in 2025, it's also growing faster than revenue (8.2x vs 7.8x), which means the company is getting more labour-heavy as it scales.
2. Despite revenue growing almost 8x, overall margin has stayed steady at around 44%. Growth hasn't hurt profitability.
3. Switching the filter to NPO shows a completely different picture: margins of only 10 to 25%, compared to ~44% for Private and Govt clients.

### Graph 3

In [27]:
sector_df = (
    RC
    .merge(profiles, on="CLIENT ID", how="left")
    .merge(survey, on=["CLIENT ID", "YEAR"], how="left")
)

In [28]:
sector_order = (sector_df.groupby("SECTOR")["gross_margin_pct"].median()
                  .sort_values().index.tolist())
n_by_sector = sector_df["SECTOR"].value_counts()
sector_labels = {s: f"{s}<br>(n={n_by_sector[s]})" for s in sector_order}
sector_df["sector_label"] = sector_df["SECTOR"].map(sector_labels)
sector_label_order = [sector_labels[s] for s in sector_order]

sector_df["point_text"] = sector_df["CLIENT ID"] + " · " + sector_df["YEAR"].astype(str)

In [29]:
overall_margin = sector_df["gross_profit"].sum() / sector_df["REVENUE"].sum() * 100

metrics = {
    "Gross Margin %": dict(col="gross_margin_pct", axis_type="linear",
                           fmt=".1f", suffix="%", refline=overall_margin),
    "Revenue":        dict(col="REVENUE", axis_type="log",   # log: revenue is heavily skewed
                           fmt=",.0f", suffix=" $", refline=None),
    "NPS Rating":     dict(col="NPS RATING", axis_type="linear",
                           fmt=".0f", suffix="", refline=None),
}
default_metric = next(iter(metrics))                 # first entry = initial view

In [30]:
charity_mask = sector_df["SECTOR"] == "Charity"
sector_groups = [
    ("Other sectors", "#7f8fa6", ~charity_mask),
    ("Charity", "#d62728", charity_mask),
]

fig = go.Figure()

for name, metric in metrics.items():
    show = name == default_metric              # initial view
    for group_name, group_colour, group_mask in sector_groups:
        subset = sector_df[group_mask]
        fig.add_trace(go.Box(
            x=subset["sector_label"], y=subset[metric["col"]],
            name=group_name,
            marker_color=group_colour,
            boxpoints="all", jitter=0.5, pointpos=0,   # points ON the box, jittered
            marker=dict(size=4, opacity=0.5),
            line=dict(width=1.5),
            visible=show,
            text=subset["point_text"],
            hovertemplate="%{text}<br>%{y:" + metric["fmt"] + "}" + metric["suffix"],
        ))

In [31]:
def ref_shapes(metric):
    if metric["refline"] is None:
        return []
    return [dict(type="line", xref="paper", x0=0, x1=1,
                 y0=metric["refline"], y1=metric["refline"],
                 line=dict(dash="dot", color="#555", width=2))]

buttons = [dict(label=name, method="update",
                args=[{"visible": [n == name for n in metrics for _ in range(len(sector_groups))]},
                      {"yaxis.title.text": name,
                       "yaxis.type": metric["axis_type"],
                       "shapes": ref_shapes(metric),
                       "title.text": f"Distribution of {name} by sector"}])
           for name, metric in metrics.items()]

In [32]:
fig.update_layout(
    title=f"Distribution of {default_metric} by sector",
    xaxis=dict(title="Sector (ordered by median margin, worst to best)",
               categoryorder="array", categoryarray=sector_label_order),
    yaxis=dict(title=default_metric),
    shapes=ref_shapes(metrics[default_metric]),
    updatemenus=[dict(type="buttons", direction="right",   # renders as a radio row
                      buttons=buttons, x=0.5, xanchor="center",
                      y=1.12, yanchor="top", showactive=True)],
    boxmode="group",
    legend=dict(orientation="h", y=-0.25),
    template="plotly_white",
    height=620,
    margin=dict(t=110),
)

fig.show()

### Insights
1. In terms of profits, the charity sector's entire margin distribution sits below every other sector (median 18.6% vs 41.8–47.3% for the rest).
2. In revenue, the charity sector is financially tiny (around $1.0M of $135M total, less than 1% of total revenue). So the losses are small in dollar terms, and the company can decide calmly whether to keep supporting these clients cheaply or start charging them properly.
3. In terms of NPS, charity flips to one of the highest-rated sectors (median 9). Confirming that the clients who like us most are the ones we earn the least from.

## DB 2 (Zafran)

#### Graph 1

Dimension vs NPS rating boxplot

In [33]:
"""
Dimension Score vs NPS Rating — grouped boxplots, px only
One box per (dimension, score) combination, all on a single chart.
Use the legend as a checkbox: click any dimension's legend entry to
show/hide just that dimension's boxes (multiple can be hidden at once,
independently — this is Plotly Express's native legend-click behavior).
"""

import pandas as pd
import plotly.express as px

# ---------------------------------------------------------------------------
# Load & reshape to long format: one row per (client, year, dimension, score)
# ---------------------------------------------------------------------------
survey = pd.read_csv("cleaned/client_survey_clean.csv")

DIMENSIONS = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT",
]

long_df = survey.melt(
    id_vars=["YEAR", "CLIENT ID", "NPS RATING", "nps_category"],
    value_vars=DIMENSIONS,
    var_name="dimension",
    value_name="score",
)

# ---------------------------------------------------------------------------
# Grouped boxplot: x = score (1-5), y = NPS Rating, color = dimension
# ---------------------------------------------------------------------------
fig = px.box(
    long_df,
    x="score",
    y="NPS RATING",
    color="dimension",
    points="outliers",  # set to "all" if you want every individual data point shown too
    category_orders={"dimension": DIMENSIONS, "score": [1, 2, 3, 4, 5]},
    title="Dimension Score vs NPS Rating — click legend items to include/exclude",
    labels={"score": "Dimension Score (1–5)", "NPS RATING": "NPS Rating"},
)

fig.update_layout(
    boxmode="group",
    template="plotly_white",
    legend_title="Dimension (click to toggle)",
    xaxis=dict(dtick=1),
)

fig.show()

#### Graph 2

NPS CATEGORY BAR CHART 

The ratio of nps category in a sector + ratio of npc category in all sectors in a given year

In [34]:
# merged_df1 = pd.merge(profiles, survey, on='CLIENT ID', how='inner')
# NPSPerYEAR = merged_df1.groupby(['SECTOR','YEAR'])['nps_category'].value_counts().reset_index()
# NPSPerYEAR

In [35]:
# fig = go.Figure()

# for cat in NPSPerYEAR['nps_category'].unique():
#     subset = NPSPerYEAR[NPSPerYEAR['nps_category'] == cat]
#     fig.add_trace(
#         go.Bar(
#             x=subset['YEAR'],
#             y=subset['count'],
#             name=cat
#         )
#     )

# fig.update_layout(
#     title='Proportions of different NPS categories',
#     xaxis_title='Year',
#     yaxis_title='Counts of Each Category',
#     barmode = 'stack',
#     height=1000
# )

# fig.show()

In [36]:
"""
NPS Category Share by Sector + All-Sector Benchmark — with a Year Slider
Stacked, %-normalized bar chart. One bar per sector, plus an "All Sectors"
bar for the overall benchmark, for each year. Drag the slider to see how
the mix shifts year over year.
"""

import pandas as pd
import plotly.graph_objects as go

# ---------------------------------------------------------------------------
# Load & merge
# ---------------------------------------------------------------------------
profiles = pd.read_csv("cleaned/client_profiles_clean.csv")
survey = pd.read_csv("cleaned/client_survey_clean.csv")
df = survey.merge(profiles[["CLIENT ID", "SECTOR", "org_size"]], on="CLIENT ID", how="left")

NPS_ORDER = ["Detractor", "Passive", "Promoter"]
NPS_COLORS = {"Detractor": "#E4572E", "Passive": "#F3A712", "Promoter": "#2E933C"}
years = sorted(df["YEAR"].unique())

# Fix the sector order using overall frequency across all years (most clients first),
# so the x-axis categories don't jump around as the slider moves
sector_order = (
    df["SECTOR"].value_counts().index.tolist()
)
x_categories = sector_order + ["All Sectors"]

def shares_for_year(year):
    """Return a DataFrame indexed by x_categories with Detractor/Passive/Promoter % columns."""
    sub = df[df["YEAR"] == year]

    # per-sector counts
    counts = sub.groupby(["SECTOR", "nps_category"]).size().unstack(fill_value=0)
    counts = counts.reindex(sector_order, fill_value=0)
    for cat in NPS_ORDER:
        if cat not in counts.columns:
            counts[cat] = 0
    counts = counts[NPS_ORDER]

    # all-sectors row (overall benchmark for that year)
    overall = sub["nps_category"].value_counts()
    overall_row = pd.Series({cat: overall.get(cat, 0) for cat in NPS_ORDER}, name="All Sectors")
    counts = pd.concat([counts, overall_row.to_frame().T])

    totals = counts.sum(axis=1).replace(0, pd.NA)
    shares = counts.div(totals, axis=0) * 100
    return shares.reindex(x_categories).fillna(0)

# ---------------------------------------------------------------------------
# Build frames — one per year
# ---------------------------------------------------------------------------
frames = []
for yr in years:
    shares = shares_for_year(yr)
    frame_data = [
        go.Bar(
            x=x_categories,
            y=shares[cat],
            marker_color=NPS_COLORS[cat],
            text=[f"{v:.0f}%" for v in shares[cat]],
            textposition="inside",
            name=cat,
        )
        for cat in NPS_ORDER
    ]
    frames.append(go.Frame(data=frame_data, name=str(yr)))

# ---------------------------------------------------------------------------
# Initial figure = first year's frame
# ---------------------------------------------------------------------------
initial_shares = shares_for_year(years[0])
fig = go.Figure(
    data=[
        go.Bar(
            x=x_categories,
            y=initial_shares[cat],
            marker_color=NPS_COLORS[cat],
            text=[f"{v:.0f}%" for v in initial_shares[cat]],
            textposition="inside",
            name=cat,
        )
        for cat in NPS_ORDER
    ],
    frames=frames,
)

# Highlight the "All Sectors" bar with a border so it reads as the benchmark, not just another sector
fig.update_traces(marker_line_width=0)
for cat_idx in range(len(NPS_ORDER)):
    line_widths = [0] * len(sector_order) + [3]
    line_colors = ["rgba(0,0,0,0)"] * len(sector_order) + ["black"]
    fig.data[cat_idx].marker.line.width = line_widths
    fig.data[cat_idx].marker.line.color = line_colors

# ---------------------------------------------------------------------------
# Slider + play/pause controls
# ---------------------------------------------------------------------------
slider_steps = [
    dict(
        method="animate",
        args=[[str(yr)], {"mode": "immediate", "frame": {"duration": 300, "redraw": True}, "transition": {"duration": 200}}],
        label=str(yr),
    )
    for yr in years
]

fig.update_layout(
    barmode="stack",
    title=f"NPS Category Share by Sector vs. All-Sector Benchmark — {years[0]} to {years[-1]}",
    xaxis_title="Sector",
    yaxis_title="Share of Clients (%)",
    yaxis=dict(ticksuffix="%", range=[0, 100]),
    legend_title="NPS Category",
    template="plotly_white",
    sliders=[dict(
        active=0,
        currentvalue={"prefix": "Year: "},
        steps=slider_steps,
        pad={"t": 40},
    )],
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        y=1.15, x=1.05, xanchor="right",
        buttons=[
            dict(label="Play", method="animate",
                 args=[None, {"frame": {"duration": 600, "redraw": True}, "fromcurrent": True, "transition": {"duration": 200}}]),
            dict(label="Pause", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
        ],
    )],
)

# Update the title as the slider moves
for i, yr in enumerate(years):
    frames[i].layout = go.Layout(title=f"NPS Category Share by Sector vs. All-Sector Benchmark — {yr}")

fig.show()

C:\Users\chang\AppData\Local\Temp\ipykernel_10092\3428313992.py:48: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\chang\AppData\Local\Temp\ipykernel_10092\3428313992.py:48: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\chang\AppData\Local\Temp\ipykernel_10092\3428313992.py:48: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

#### Graph 3

NPS Rating distribution by SECTOR violin

In [37]:
"""
NPS Rating Violin — dropdown to pick SECTOR, slider to pick YEAR
Two implementations: graph_objects (fully custom) and express (px.violin
under the hood, with a dropdown bolted on top for sector switching).

Note: a few sectors have very thin sample sizes in early years
(e.g. Finance = 1 client every year, Legal/Education = 0 before 2023).
In those years the violin will collapse to a single point or be empty —
that's expected, not a bug. Hover/points still show the raw values.
"""

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# ---------------------------------------------------------------------------
# Load & merge
# ---------------------------------------------------------------------------
profiles = pd.read_csv("cleaned/client_profiles_clean.csv")
survey = pd.read_csv("cleaned/client_survey_clean.csv")
df = survey.merge(profiles[["CLIENT ID", "SECTOR", "org_size"]], on="CLIENT ID", how="left")

years = sorted(df["YEAR"].unique())
sector_order = df["SECTOR"].value_counts().index.tolist()  # most-represented first
sectors_all = ["All Sectors"] + sector_order

VIOLIN_COLOR = "#4C78A8"


def data_for(sector, year):
    """Return the NPS RATING series for a given sector/year (or all sectors)."""
    sub = df[df["YEAR"] == year]
    if sector != "All Sectors":
        sub = sub[sub["SECTOR"] == sector]
    return sub["NPS RATING"]


# ===========================================================================
# GO VERSION — dropdown (sector) + slider (year), fully custom
# ===========================================================================
# Strategy: pre-build one frame per (sector, year) combo. The slider's steps
# reference frames for the CURRENTLY selected sector; picking a new sector
# from the dropdown swaps the trace data AND rewrites the slider's steps to
# point at that sector's frames.

frames = []
for sector in sectors_all:
    for yr in years:
        vals = data_for(sector, yr)
        frames.append(
            go.Frame(
                data=[go.Violin(
                    y=vals,
                    x=[str(yr)] * len(vals),
                    name=str(yr),
                    box_visible=True,
                    meanline_visible=True,
                    points="all",
                    pointpos=0,
                    jitter=0.3,
                    marker=dict(size=4, opacity=0.5, color=VIOLIN_COLOR),
                    fillcolor=VIOLIN_COLOR,
                    opacity=0.6,
                    line_color=VIOLIN_COLOR,
                )],
                name=f"{sector}__{yr}",
                layout=go.Layout(title=f"NPS Rating Distribution — {sector} — {yr}"),
            )
        )

def steps_for_sector(sector):
    return [
        dict(
            method="animate",
            args=[[f"{sector}__{yr}"], {"mode": "immediate",
                                         "frame": {"duration": 300, "redraw": True},
                                         "transition": {"duration": 200}}],
            label=str(yr),
        )
        for yr in years
    ]

def initial_trace(sector, year):
    vals = data_for(sector, year)
    return go.Violin(
        y=vals,
        x=[str(year)] * len(vals),
        name=str(year),
        box_visible=True,
        meanline_visible=True,
        points="all",
        pointpos=0,
        jitter=0.3,
        marker=dict(size=4, opacity=0.5, color=VIOLIN_COLOR),
        fillcolor=VIOLIN_COLOR,
        opacity=0.6,
        line_color=VIOLIN_COLOR,
    )

DEFAULT_SECTOR = "All Sectors"
DEFAULT_YEAR = years[0]

fig_go = go.Figure(
    data=[initial_trace(DEFAULT_SECTOR, DEFAULT_YEAR)],
    frames=frames,
)

dropdown_buttons = []
for sector in sectors_all:
    trace0 = initial_trace(sector, years[0])
    dropdown_buttons.append(
        dict(
            label=sector,
            method="update",
            args=[
                # --- data update (restyle) ---
                {
                    "y": [trace0.y],
                    "x": [trace0.x],
                    "name": [trace0.name],
                },
                # --- layout update (relayout) ---
                {
                    "sliders[0].steps": steps_for_sector(sector),
                    "sliders[0].active": 0,
                    "title": f"NPS Rating Distribution — {sector} — {years[0]}",
                },
            ],
        )
    )

fig_go.update_layout(
    title=f"NPS Rating Distribution — {DEFAULT_SECTOR} — {DEFAULT_YEAR}",
    yaxis_title="NPS Rating",
    xaxis_title="Year",
    template="plotly_white",
    showlegend=False,
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            x=1.05, xanchor="left",
            y=1.15, yanchor="top",
            showactive=True,
        )
    ],
    sliders=[
        dict(
            active=0,
            currentvalue={"prefix": "Year: "},
            steps=steps_for_sector(DEFAULT_SECTOR),
            pad={"t": 40},
        )
    ],
)
fig_go.show()

## DB 3 (name)

#### Graph 1

#### Graph 2

#### Graph 3